# Step 3 — On-Device Speech Transcription
**Tough Talks · Phase 2**

Goal: prove out **Gemma 4 E2B's native audio path** end-to-end on Colab/Kaggle.
We reuse the same multimodal model that will power emotion + reasoning so the
Live-Mode stack stays single-model. The notebook is a thin driver — all logic
lives in `backend/core/_runtime/audio.py`.

**What "done" looks like for this step**
1. Multimodal Gemma 4 loads (`AutoModelForMultimodalLM` via `LoadConfig(multimodal=True)`).
2. A short canned WAV (≤ 30 s, downloaded at runtime, gitignored) is transcribed.
3. The output dict validates against `data/schemas/transcription.schema.json`.

**Gemma 4 audio constraints worth remembering**
- Audio content must come **before** the text instruction in `content`.
- Max clip length is **30 seconds** (`MAX_AUDIO_SECONDS`); chunk longer audio upstream.
- Audio support is on **E2B and E4B only** (other Gemma 4 sizes are text/vision).

In [ ]:
# ── 0. Install / upgrade dependencies ────────────────────────────────────────
# Phase 2 adds audio I/O on top of Phase 1's transformers/accelerate stack.
# librosa + soundfile are what the AutoProcessor's audio feature extractor
# uses under the hood to read the WAV.
#
# DO NOT bump torch on Colab/Kaggle — it breaks the pre-installed
# torchvision/CUDA pairing. Only bump transformers + accelerate + audio libs.
# After this cell runs once, RESTART THE KERNEL before re-running anything,
# otherwise the already-imported transformers module won't pick up the upgrade.

!pip install -q -U transformers accelerate librosa soundfile

In [ ]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────────
# Same shim as Step 1 — auto-clones / refreshes on Colab / Kaggle.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import json
import urllib.request
from pathlib import Path

import torch

from backend.core._runtime import (
    DEFAULT_MODEL_ID,
    LoadConfig,
    MAX_AUDIO_SECONDS,
    TranscribeConfig,
    load_model,
    transcribe,
)

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────────────

MODEL_ID = DEFAULT_MODEL_ID                                # google/gemma-4-E2B-it
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# Canonical short clip from the official Gemma cookbook (≤ 30 s spoken English).
# Downloaded at runtime into REPO_ROOT/data/audio_cache/ which is gitignored
# (the global *.wav rule plus audio_cache/ both cover it).
AUDIO_URL    = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/journal1.wav"
AUDIO_DIR    = Path(REPO_ROOT) / "data" / "audio_cache"
AUDIO_LOCAL  = AUDIO_DIR / "journal1.wav"

SCHEMA_PATH  = Path(REPO_ROOT) / "data" / "schemas" / "transcription.schema.json"

print(f"Model   : {MODEL_ID}")
print(f"Device  : {DEVICE}")
print(f"Audio   : {AUDIO_LOCAL}")
print(f"Max clip: {MAX_AUDIO_SECONDS}s (Gemma 4 audio context cap)")

In [ ]:
# ── 4. Fetch the canned sample clip (gitignored) ─────────────────────────────
# Idempotent: skip if already cached. The URL serves a short journal-style
# spoken English clip that ships with the official Gemma 4 cookbook, so we
# can compare model output against a known transcript.

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

if AUDIO_LOCAL.exists():
    print(f"Audio already cached: {AUDIO_LOCAL} ({AUDIO_LOCAL.stat().st_size/1024:.1f} KB)")
else:
    print(f"Downloading {AUDIO_URL}")
    urllib.request.urlretrieve(AUDIO_URL, AUDIO_LOCAL)
    print(f"Saved {AUDIO_LOCAL} ({AUDIO_LOCAL.stat().st_size/1024:.1f} KB)")

# Quick duration probe — fails loud if the clip exceeds Gemma 4's 30s cap.
import soundfile as sf
info = sf.info(str(AUDIO_LOCAL))
DURATION_S = info.frames / info.samplerate
print(f"Duration: {DURATION_S:.2f}s @ {info.samplerate} Hz, {info.channels}ch")
assert DURATION_S <= MAX_AUDIO_SECONDS, f"Clip is {DURATION_S:.1f}s — exceeds Gemma 4's {MAX_AUDIO_SECONDS}s cap"

In [ ]:
# ── 5. Load the multimodal processor + model ─────────────────────────────────
# multimodal=True swaps AutoModelForCausalLM for AutoModelForMultimodalLM so
# the audio inputs in apply_chat_template() are actually consumed by the model.

processor, model = load_model(LoadConfig(model_id=MODEL_ID, multimodal=True))

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

In [ ]:
# ── 6. Transcribe ────────────────────────────────────────────────────────────
# transcribe() builds the [audio-first, then text] message list, calls
# apply_chat_template with tokenize/return_dict/return_tensors, slices off the
# prompt tokens, decodes with parse_response, and packages the result into a
# schema-conforming dict.

cfg = TranscribeConfig(
    max_new_tokens=256,
    language="en",
    duration_seconds=round(DURATION_S, 2),
)

result = transcribe(processor, model, AUDIO_LOCAL, cfg=cfg, model_id=MODEL_ID)
print(json.dumps(result, indent=2))

In [ ]:
# ── 7. Schema validation (no third-party deps) ───────────────────────────────
# Hand-checks the contract instead of pulling jsonschema in. Covers:
#   * required keys present
#   * field types match
#   * no surprise top-level keys (additionalProperties: false)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))

_TYPE_MAP = {
    "string": (str,),
    "number": (int, float),
    "boolean": (bool,),
    "integer": (int,),
    "object": (dict,),
    "array": (list,),
    "null": (type(None),),
}

def _validate_against_schema(payload: dict, schema: dict) -> list[str]:
    errors: list[str] = []
    required = schema.get("required", [])
    props = schema.get("properties", {})
    for key in required:
        if key not in payload:
            errors.append(f"missing required field: {key!r}")
    for key, value in payload.items():
        if key not in props:
            if schema.get("additionalProperties") is False:
                errors.append(f"unexpected top-level field: {key!r}")
            continue
        spec = props[key]
        declared = spec.get("type")
        types = [declared] if isinstance(declared, str) else (declared or [])
        if not types:
            continue
        allowed = tuple(t for name in types for t in _TYPE_MAP.get(name, ()))
        if allowed and not isinstance(value, allowed):
            errors.append(f"{key!r}: expected {types}, got {type(value).__name__}")
    return errors

schema_errors = _validate_against_schema(result, schema)
if schema_errors:
    print("SCHEMA ERRORS:")
    for e in schema_errors:
        print(f"  - {e}")
else:
    print("Schema validation: OK")

In [ ]:
# ── 8. Step 3 results table ──────────────────────────────────────────────────
# Renders unconditionally — same pattern as Step 1 so partial failures are
# visible at a glance.

checks: list[tuple[str, bool, str]] = [
    ("multimodal_model_loaded", True, f"{n_params:.1f}B params on {model.device}"),
    ("audio_clip_under_30s",    DURATION_S <= MAX_AUDIO_SECONDS, f"{DURATION_S:.2f}s"),
    ("transcript_non_empty",    bool(result.get("transcript")), f"{len(result.get('transcript', ''))} chars"),
    ("schema_valid",            not schema_errors, "ok" if not schema_errors else f"{len(schema_errors)} error(s)"),
]

print("=" * 66)
print("STEP 3 RESULTS — Gemma 4 native speech transcription")
print("=" * 66)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

print()
print("Transcript preview:")
print(f"  {result.get('transcript', '')[:240]}")
print()
print("OVERALL:", "READY FOR STEP 4" if all_ok else "FIX FAILURES ABOVE")